# Initial clustering run

This notebook will contain the initial clustering run with one fixed aggregation level and window size,choosing of optimal k, saving the model, and computing the ARI metric for consecutive weeks

## Data & Packages

In [72]:
# Main packages 
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# Clustering packages
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score,adjusted_rand_score

# Parallel processing packages
from joblib import Parallel, delayed

In [73]:
# Load data
df = pl.scan_csv("/home/lanl/data/cyber1/auth.txt.gz",has_header=False,separator=",",
                 new_columns= ['time','src_user','dest_user','src_comp','dest_comp',
                               'auth_type','logon_type','auth_orientation','outcome'])

In [74]:
# Keep only human users
df = df.filter(pl.col("src_user").str.starts_with("U"))

## Functions & Global info

In [ ]:
# Time conversions
SECONDS_IN_DAY = 60 * 60 * 24

# Time aggregation
agg_hour_level = 1

# Number of buckets in a week of data
buckets_per_week = (7 * 24) // agg_hour_level

In [76]:
# FUNCTION - build features dataframe
def build_features(df,agg_hour_level):

    AGG_SECONDS = agg_hour_level * 60 * 60
    return (
        df.with_columns(
            bucket = pl.col('time') // AGG_SECONDS,
            theta = ((pl.col('time') % SECONDS_IN_DAY)/ SECONDS_IN_DAY) * 2 * np.pi,
            is_failure = (pl.col('outcome') == 'Fail').cast(pl.Int8),
        )
        .group_by(['src_user','bucket'])
        .agg(
            n_events = pl.len(),
            failure_ratio = pl.col('is_failure').mean(),
            n_distinct_dest = pl.col('dest_comp').n_unique(),
            n_distinct_src = pl.col('src_comp').n_unique(),
            c_bar = pl.col('theta').cos().mean(),
            s_bar = pl.col('theta').sin().mean(),
        )
        .with_columns(
             log_n_events=pl.col("n_events").log1p(),
             log_n_distinct_dest=pl.col("n_distinct_dest").log1p(),
             log_n_distinct_src=pl.col("n_distinct_src").log1p(),
        )
        .collect()
    )

In [77]:
# Relevant feauture columns
feature_cols = [
    "log_n_events",
    "failure_ratio",
    "log_n_distinct_dest",
    "log_n_distinct_src",
    "c_bar",
    "s_bar",
]

In [ ]:
# FUNCTION - process features for clustering 
def cluster_preprocess(features_df,feature_cols,agg_hour_level,week):

    buckets_per_week = (7 * 24) // agg_hour_level
    lb = (week - 1) * buckets_per_week
    ub = lb + buckets_per_week - 1

    features_week = features_df.filter(pl.col('bucket').is_between(lb,ub))

    X = features_week.select(feature_cols).to_numpy()
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return features_week, X_scaled

In [79]:
# FUNCTION - kmeans 
def fit_kmeans(k, Y, sample_size):
    km = KMeans(n_clusters=k, random_state=123, n_init=10)
    labels = km.fit_predict(Y)
    
    sil = silhouette_score(Y, labels, sample_size=sample_size, random_state=123)
    ch  = calinski_harabasz_score(Y, labels)   
    db  = davies_bouldin_score(Y, labels)    
    
    return k, sil,ch,db

In [80]:
# Create features dataframe
features_df = build_features(df,agg_hour_level)

## Week 1 Analysis

In [81]:
# Process features for target week
features_w1, X_scaled = cluster_preprocess(features_df,feature_cols,agg_hour_level,week = 1)

In [82]:
# Run kmeans in parallel
sample_size = 1000
results = Parallel(n_jobs=-1)(delayed(fit_kmeans)(k, X_scaled,sample_size) for k in range(2, 11))

In [83]:
# Best result
best_result = max(results, key = lambda row: row[1])
best_k, best_score  = best_result[0], best_result[1]

print(f'optimal number of clusters is k = {best_k}')
print(f'silhouette score = {best_score}')

optimal number of clusters is k = 2
silhouette score = 0.32420599167863734


In [84]:
# Refit k-means on best result
km = KMeans(n_clusters=best_k, random_state=123, n_init=10)
labels = km.fit_predict(X_scaled)

In [85]:
# Attach labels back to the dataframe
features_w1 = (features_w1
    .with_columns(pl.Series("cluster", labels))
    .select(['src_user','bucket','cluster'])
)

In [86]:
# Cluster breakdown
features_w1['cluster'].value_counts()

cluster,count
i32,u64
0,267064
1,527325


In [87]:
# Save results
features_w1.write_parquet("/home/ma/t/tr725/MSc-Project/03_working_notebooks/results/week1.parquet")

## Week 2 Analysis

In [88]:
# Process features for target week
features_w2, X_scaled = cluster_preprocess(features_df,feature_cols,agg_hour_level,week = 2)

In [89]:
# Run kmeans in parallel
sample_size = 1000
results = Parallel(n_jobs=-1)(delayed(fit_kmeans)(k, X_scaled,sample_size) for k in range(2, 11))

In [90]:
# Best result
best_result = max(results, key = lambda row: row[1])
best_k, best_score  = best_result[0], best_result[1]

print(f'optimal number of clusters is k = {best_k}')
print(f'silhouette score = {best_score}')

optimal number of clusters is k = 2
silhouette score = 0.3160506972616562


In [91]:
# Refit k-means on best result
km = KMeans(n_clusters=best_k, random_state=123, n_init=10)
labels = km.fit_predict(X_scaled)

In [92]:
# Attach labels back to the dataframe
features_w2 = (features_w2
    .with_columns(pl.Series("cluster", labels))
    .select(['src_user','bucket','cluster']))

In [93]:
# Cluster breakdown
features_w2['cluster'].value_counts()

cluster,count
i32,u64
0,603509
1,299905


In [94]:
# Save results
features_w2.write_parquet("/home/ma/t/tr725/MSc-Project/03_working_notebooks/results/week2.parquet")

## Week 3 Analysis

In [110]:
# Process features for target week
features_w3, X_scaled = cluster_preprocess(features_df,feature_cols,agg_hour_level,week = 3)

In [111]:
# Run kmeans in parallel
sample_size = 1000
results = Parallel(n_jobs=-1)(delayed(fit_kmeans)(k, X_scaled,sample_size) for k in range(2, 11))

In [112]:
# Best result
best_result = max(results, key = lambda row: row[1])
best_k, best_score  = best_result[0], best_result[1]

print(f'optimal number of clusters is k = {best_k}')
print(f'silhouette score = {best_score}')

optimal number of clusters is k = 3
silhouette score = 0.3299564556698617


In [113]:
# Refit k-means on best result
km = KMeans(n_clusters=best_k, random_state=123, n_init=10)
labels = km.fit_predict(X_scaled)

In [114]:
# Attach labels back to the dataframe
features_w3 = (features_w3
    .with_columns(pl.Series("cluster", labels))
    .select(['src_user','bucket','cluster']))

In [125]:
# Cluster breakdown
features_w3['cluster'].value_counts()

cluster,count
i32,u64
1,257426
0,514282
2,24537


In [116]:
# Save results
features_w3.write_parquet("/home/ma/t/tr725/MSc-Project/03_working_notebooks/results/week3.parquet")

# Adjusted Rand Index

## Week 1 - 2

In [ ]:
# Processing for clustering stability
features_w1 = features_w1.with_columns(
    relative_bucket = pl.col('bucket') % buckets_per_week
) 

features_w2 = features_w2.with_columns(
    relative_bucket = pl.col('bucket') % buckets_per_week
) 

overlap = features_w1.join(
    features_w2,
    on = ['src_user','relative_bucket'],
    how = 'inner',
    suffix = '_w2'
)

labels_w1 = overlap['cluster'].to_numpy()
labels_w2 = overlap['cluster_w2'].to_numpy()

In [109]:
# Adjusted Rand Index
ARI = adjusted_rand_score(labels_w1, labels_w2)
print(ARI)

0.10382445907015721


In [104]:
print(f"Week 1 (user, rel_bucket) pairs:  {len(features_w1)}")
print(f"Week 2 (user, rel_bucket) pairs:  {len(features_w2)}")
print(f"Overlapping pairs:                {len(overlap)}")
print(f"Overlap as % of week 1:           {len(overlap)/len(features_w1):.2%}")
print(f"Overlap as % of week 2:           {len(overlap)/len(features_w2):.2%}")

Week 1 (user, rel_bucket) pairs:  794389
Week 2 (user, rel_bucket) pairs:  903414
Overlapping pairs:                615624
Overlap as % of week 1:           77.50%
Overlap as % of week 2:           68.14%


## Week 2-3

In [133]:
# Processing for clustering stability
features_w3 = features_w3.with_columns(
    relative_bucket = pl.col('bucket') % buckets_per_week
) 

overlap = features_w2.join(
    features_w3,
    on = ['src_user','relative_bucket'],
    how = 'inner',
    suffix = '_w3'
)

labels_w2 = overlap['cluster'].to_numpy()
labels_w3 = overlap['cluster_w3'].to_numpy()

In [134]:
# Adjusted Rand Index
ARI = adjusted_rand_score(labels_w2, labels_w3)
print(ARI)

0.09118859993456639


In [135]:
print(f"Week 2 (user, rel_bucket) pairs:  {len(features_w2)}")
print(f"Week 3 (user, rel_bucket) pairs:  {len(features_w3)}")
print(f"Overlapping pairs:                {len(overlap)}")
print(f"Overlap as % of week 2:           {len(overlap)/len(features_w2):.2%}")
print(f"Overlap as % of week 3:           {len(overlap)/len(features_w3):.2%}")

Week 2 (user, rel_bucket) pairs:  903414
Week 3 (user, rel_bucket) pairs:  796245
Overlapping pairs:                626524
Overlap as % of week 2:           69.35%
Overlap as % of week 3:           78.68%


---

### Trade-offs between aggregation levels:

Taking a slight step back and understanding why we are looking at different aggregation levels, and the importance of the choice for downstream anomaly detection.

**Fined-grained**
- Captures within day behavioural rhythms and hourly patterns
- Better seperates users with similar day profiles but different hourly profiles
- Higher senstivity to 'outlier' - anomalous event's aren't diluted by surrounding normal activity
- More data points, higher computational cost
- Can generate potentially more false alarms

**Coarse-grained**
- smoothing 
- users become harder to distinguish - heterogeneuous users may merge into the same cluster 
- short duration attacks may become invisible 
- anomalies appearing at dubious times may be smoothed over
- fewer data points, cheaper to compute 

### Research Question

The practical motivation is a real engineering decision: what aggregation level should a deployed anomaly detection system use. There is no theoretical answer - it depends on the data. The research provides empirical evidence on real authentication data (LANL).

The underlying tension being resolved:

- Fine aggregation → higher sensitivity to attacks
- Coarse aggregation → lower false alarm rate, more stable behaviour model

The contribution is measuring what the aggregation dial actually does, rather than assuming finer is always better or coarser is always more stable.

### Experimental design

The experiment is not comparing stability scalar across different aggregation levels. Instead:

For each aggregation level independently, compute a time series of stability metrics (e.g ARI) values across consecutive windows. This trajectory is the primary object of analysis.

Can compare the behaviour of the trajectories:
- mean & variance
- Coincidence — do all levels show instability at the same points? If yes, suggests a genuine population-level behavioural shift. If only fine levels show it, more likely noise.